<a href="https://colab.research.google.com/github/TKWaititu/Job-finder/blob/main/job_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##This notebook outlines a python program that scrapes the job board https://www.corporatestaffing.co.ke/ and filters the jobs by looking for select keywords in the job titles and descriptions, motivated by the tedious process of going through the website page by page looking for matching jobs.



In [1]:
# libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

pd.set_option('display.max_rows', None)


In [2]:
def job_finder(page_url):
  # processing listings page
  page = requests.get(page_url)
  soup = BeautifulSoup(page.text)
  page_list =[]

  # finding all the jobs listed in page
  for job in soup.find_all('li', class_ = 'entry-list-item'):
    title_tag = job.find('h2',class_ = 'entry-title').a
    job_dict = {
                'job_title': title_tag.get_text(strip=True),
                'job_link': title_tag['href'],
                'date': job.find('time')['datetime']
                }
    page_list.append(job_dict)


  return page_list


In [3]:
def job_parser(job_url):
  # processing a particular job's webpage
  job_page = requests.get(job_url)
  job_soup = BeautifulSoup(job_page.text, "html.parser")

  # scraping the job description
  job_ul = job_soup.find_all('ul', class_='wp-block-list')

  job_description = []

  for ul in job_ul:
    for li in ul.find_all('li'):
        job_description.append(li.text)

  return job_description

In [4]:
def job_merger(first_page, last_page, search_terms):
  mega_list = []
  jobs_found = 0

  # processing a range of pages, finding the jobs and extracting the description
  for i in range(first_page, last_page + 1):
    page_url = f"https://www.corporatestaffing.co.ke/jobs/page/{i}"
    jobs_in_page = job_finder(page_url)
    jobs_found += len(jobs_in_page)

    for job in jobs_in_page:
      job_link = job['job_link']
      job_desc = job_parser(job_link)
      job['description'] = job_desc
      mega_list.append(job)


  # progress tracker
    print(f'Processed page {i} out of {last_page}')

  # converting the acquired data into a dataframe for cleaning
  df = pd.DataFrame(mega_list)

  # converting the list items into blocks of text
  df['description'] = df['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else x
)

  # removing extra whitspaces and punctuation marks
  df['description'] = (
      df['description']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  df['job_title'] = (
      df['job_title']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  # joining the title and description columns for a richer column
  df['full_desc'] = df['job_title'] + " " + df['description']

  # formatting the search terms and filtering the description column for related jobs
  pattern = '|'.join(search_terms)
  job_df = (df[df['full_desc'].str.contains(pattern, case=False, na=False)])

  print(f'Found {len(job_df)} matches out of {jobs_found} jobs.')

  return job_df

In [5]:
search_terms = ['data science', 'statistics', 'actuarial', 'data analyst', 'Finance', 'data analysis']
job_df = job_merger(1,6, search_terms)

Processed page 1 out of 6
Processed page 2 out of 6
Processed page 3 out of 6
Processed page 4 out of 6
Processed page 5 out of 6
Processed page 6 out of 6
Found 21 matches out of 120 jobs.


In [6]:
job_df

,job_title,job_link,date,description,full_desc
0,Senior Manager Youth Banking Job KCB Bank,https://www.corporatestaffing.co.ke/job/senior...,2026-03-19T17:52:00+03:00,The role focuses on developing and delivering ...,Senior Manager Youth Banking Job KCB Bank The ...
3,General Manager Job 50K,https://www.corporatestaffing.co.ke/job/genera...,2026-03-19T17:39:56+03:00,Lead and inspire teams across Sales Procuremen...,General Manager Job 50K Lead and inspire teams...
4,Branch Manager Micro Finance Job 45K,https://www.corporatestaffing.co.ke/job/branch...,2026-03-19T17:32:40+03:00,Oversee daily operations of the branch ensurin...,Branch Manager Micro Finance Job 45K Oversee d...
22,Supply Chain Assistant NGO Job Save the Children,https://www.corporatestaffing.co.ke/job/supply...,2026-03-18T17:06:00+03:00,Support the overall management of organization...,Supply Chain Assistant NGO Job Save the Child...
26,Cementing Technical Engineer Job Baker Hughes,https://www.corporatestaffing.co.ke/job/cement...,2026-03-18T17:03:30+03:00,Leading the technical preparation for cementin...,Cementing Technical Engineer Job Baker Hughes ...
27,Finance Manager Job RVIBS,https://www.corporatestaffing.co.ke/job/financ...,2026-03-18T17:02:38+03:00,Provide strategic financial leadership and adv...,Finance Manager Job RVIBS Provide strategic fi...
43,Digital Marketing Specialist Job 100130K,https://www.corporatestaffing.co.ke/job/digita...,2026-03-18T15:06:09+03:00,Bachelors degree in marketing communications b...,Digital Marketing Specialist Job 100130K Bache...
46,BD Manager Retail Distribution Job Jubilee Ins...,https://www.corporatestaffing.co.ke/job/bd-man...,2026-03-18T14:32:28+03:00,Drive acquisition of retail clients for Unit T...,BD Manager Retail Distribution Job Jubilee Ins...
50,Controller Job Solvo Global,https://www.corporatestaffing.co.ke/job/contro...,2026-03-18T14:17:23+03:00,Lead and oversee financial and accounting oper...,Controller Job Solvo Global Lead and oversee f...
51,Pension Scheme Fund Accountant Job Kenindia As...,https://www.corporatestaffing.co.ke/job/pensio...,2026-03-18T13:50:38+03:00,Ensure compliance of the pension schemes in li...,Pension Scheme Fund Accountant Job Kenindia As...


In [12]:
for title, link, date in zip(job_df['job_title'], job_df['job_link'], job_df['date']):
    print(f"Title: {title}\nLink: {link}\nDate: {date}\n")

Title: Senior Manager Youth Banking Job KCB Bank
Link: https://www.corporatestaffing.co.ke/job/senior-manager-youth-banking-job-kcb/
Date: 2026-03-19T17:52:00+03:00

Title: General Manager Job 50K
Link: https://www.corporatestaffing.co.ke/job/general-manager-job-50k-brites/
Date: 2026-03-19T17:39:56+03:00

Title: Branch Manager Micro Finance Job 45K
Link: https://www.corporatestaffing.co.ke/job/branch-manager-micro-finance-job-45k/
Date: 2026-03-19T17:32:40+03:00

Title: Supply Chain Assistant  NGO Job Save the Children
Link: https://www.corporatestaffing.co.ke/job/supply-chain-assistant-ngo-job-save-the-children/
Date: 2026-03-18T17:06:00+03:00

Title: Cementing Technical Engineer Job Baker Hughes
Link: https://www.corporatestaffing.co.ke/job/cementing-technical-engineer-job-baker-hughes/
Date: 2026-03-18T17:03:30+03:00

Title: Finance Manager Job RVIBS
Link: https://www.corporatestaffing.co.ke/job/finance-manager-job-rvibs/
Date: 2026-03-18T17:02:38+03:00

Title: Digital Marketing Sp